# Feature Engineering

Create useful features from historical sales data to predict daily product demand.

In [8]:
import pandas as pd

# Load daily sales
daily_sales = pd.read_csv("daily_product_sales.csv")
daily_sales["date"] = pd.to_datetime(daily_sales["date"])

In [9]:
# Sort data
daily_sales = daily_sales.sort_values(["article", "date"])

# Calendar features
daily_sales["day_of_week"] = daily_sales["date"].dt.dayofweek
daily_sales["month"] = daily_sales["date"].dt.month

daily_sales.head()

,date,article,Quantity,day_of_week,month
0,2021-01-02,BAGUETTE,46.0,5,1
22,2021-01-03,BAGUETTE,36.0,6,1
41,2021-01-04,BAGUETTE,30.0,0,1
61,2021-01-05,BAGUETTE,29.0,1,1
80,2021-01-07,BAGUETTE,28.0,3,1


### Historical Sales Features

Use previous sales to help predict future product demand.

In [10]:
# Previous recorded sales for each product
daily_sales["previous_sales"] = (
    daily_sales.groupby("article")["Quantity"].shift(1)
)

daily_sales.head(10)

,date,article,Quantity,day_of_week,month,previous_sales
0,2021-01-02,BAGUETTE,46.0,5,1,NaN
22,2021-01-03,BAGUETTE,36.0,6,1,46.0
41,2021-01-04,BAGUETTE,30.0,0,1,36.0
61,2021-01-05,BAGUETTE,29.0,1,1,30.0
80,2021-01-07,BAGUETTE,28.0,3,1,29.0
97,2021-01-08,BAGUETTE,17.0,4,1,28.0
117,2021-01-09,BAGUETTE,25.0,5,1,17.0
136,2021-01-10,BAGUETTE,36.0,6,1,25.0
153,2021-01-11,BAGUETTE,31.0,0,1,36.0
171,2021-01-12,BAGUETTE,29.0,1,1,31.0


### Rolling Average

Calculate the average sales of the previous 3 observations for each product to capture recent demand trends.

In [11]:
# Average sales of previous 3 observations
daily_sales["avg_last_3_sales"] = (
    daily_sales.groupby("article")["previous_sales"]
    .rolling(3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

daily_sales.head(10)

,date,article,Quantity,day_of_week,month,previous_sales,avg_last_3_sales
0,2021-01-02,BAGUETTE,46.0,5,1,NaN,NaN
22,2021-01-03,BAGUETTE,36.0,6,1,46.0,46.000000
41,2021-01-04,BAGUETTE,30.0,0,1,36.0,41.000000
61,2021-01-05,BAGUETTE,29.0,1,1,30.0,37.333333
80,2021-01-07,BAGUETTE,28.0,3,1,29.0,31.666667
97,2021-01-08,BAGUETTE,17.0,4,1,28.0,29.000000
117,2021-01-09,BAGUETTE,25.0,5,1,17.0,24.666667
136,2021-01-10,BAGUETTE,36.0,6,1,25.0,23.333333
153,2021-01-11,BAGUETTE,31.0,0,1,36.0,26.000000
171,2021-01-12,BAGUETTE,29.0,1,1,31.0,30.666667


### Prepare ML Dataset

In [12]:
# Rename target column
daily_sales = daily_sales.rename(columns={"Quantity": "daily_quantity_sold"})

daily_sales.head()

,date,article,daily_quantity_sold,day_of_week,month,previous_sales,avg_last_3_sales
0,2021-01-02,BAGUETTE,46.0,5,1,NaN,NaN
22,2021-01-03,BAGUETTE,36.0,6,1,46.0,46.000000
41,2021-01-04,BAGUETTE,30.0,0,1,36.0,41.000000
61,2021-01-05,BAGUETTE,29.0,1,1,30.0,37.333333
80,2021-01-07,BAGUETTE,28.0,3,1,29.0,31.666667


In [13]:
# Remove rows with missing historical sales
daily_sales = daily_sales.dropna(
    subset=["previous_sales", "avg_last_3_sales"]
).reset_index(drop=True)

# Check the result
print(daily_sales.isnull().sum())
print(daily_sales.shape)

date                   0
article                0
daily_quantity_sold    0
day_of_week            0
month                  0
previous_sales         0
avg_last_3_sales       0
dtype: int64
(13467, 7)


In [14]:
# Save prepared dataset
daily_sales.to_csv("ml_ready_bakery.csv", index=False)